# [B3; B1/B2/A4/F1/F3/E3/E5] STTR partner-effect feasibility

**Status:** exploratory / non-citable  
**Research question:** Can a Phase I STTR partner-variance model be preregistered with the public data currently available?  
**Decision this informs:** whether to draft the NSF experiment's gate-model specification, and which extensions remain blocked.  
**Data as of:** SBIR.gov snapshot retrieved 2026-09-07; exact SHA-256 below  
**Owner:** STTR partner-effect Phase 0

This notebook computes availability and sample counts only. It does not load outcomes by RI, estimate a partner effect, form density tiers, or run any model.

## Data contract

- **Population:** canonical SBIR.gov awards with `Program = STTR` and `Phase = Phase I` for the gate counts; all-phase STTR is used for field coverage, while Question 3 uses all canonical SBIR/STTR awards and reports a Phase I-only sensitivity.
- **Grain:** canonical award after `sbir-source-v2` edition collapse.
- **Keys:** canonical `award_key`; RI string key from the existing STTR kernel's `resolve_identity(..., ORGANIZATION)` default `MATCHING_V1` behavior. This is normalization, not institution-level alias resolution.
- **Raw-field join:** the canonical row's selected `source_row` joins `RI Name`, PI email, and proposal dates from the exact 42-field export.
- **Firm key:** existing `assign_firm_keys`; no notebook-specific firm matcher.
- **Topic/FY proxy:** exact `normalize_topic_code` plus `FiscalShockAggregator.extract_fiscal_year`. No canonical topic-family primitive exists.
- **Missingness:** blank fields and absent artifacts remain unavailable, never negative evidence.
- **Outputs:** exploratory tables displayed in memory. No Parquet, database, Neo4j, or Phase III asset write.

In [ ]:
from __future__ import annotations

import os
import re
from pathlib import Path

import numpy as np
import pandas as pd

from sbir_etl.extractors.sbir_award_export import inspect_sbir_gov_csv
from sbir_etl.extractors.sbir_public_awards import load_sbir_awards_csv
from sbir_etl.transformers.fiscal.shocks import FiscalShockAggregator
from sbir_etl.utils.data.file_io import file_sha256
from scripts.data.find_same_work_awards import assign_firm_keys, normalize_topic_code
from scripts.sttr_spinout_linkage.kernel import IdentityKind, resolve_identity


def find_repo_root(start: Path = Path.cwd()) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'pyproject.toml').exists() and (candidate / 'sbir_etl').exists():
            return candidate
    raise RuntimeError('Run this notebook from inside the sbir-analytics checkout')


REPO_ROOT = find_repo_root()
REPO_ROOT

In [ ]:
EXPECTED_SHA256 = 'aed146eab56f370c9f3fe7f562475e3eedfc61cca2eba112c830fac6f73bf38a'
EXPECTED_SOURCE_ROWS = 219_590
EXPECTED_CANONICAL_AWARDS = 219_535
AS_OF_DATE = '2026-09-07'

configured_path = os.environ.get('SBIR_AWARDS_CSV')
award_candidates = [
    Path(configured_path) if configured_path else None,
    Path('/Volumes/SSDmini/sbir-analytics/data/raw/sbir/award_data.csv'),
    REPO_ROOT / 'data' / 'raw' / 'sbir' / 'award_data.csv',
]
AWARDS_CSV = next((path for path in award_candidates if path is not None and path.is_file()), None)
if AWARDS_CSV is None:
    raise FileNotFoundError('Set SBIR_AWARDS_CSV to the snapshot pinned in this notebook')

observed_sha256 = file_sha256(AWARDS_CSV)
if observed_sha256 != EXPECTED_SHA256:
    raise RuntimeError(
        f'Wrong SBIR.gov snapshot: expected {EXPECTED_SHA256}, observed {observed_sha256}'
    )
inspection = inspect_sbir_gov_csv(AWARDS_CSV)
assert inspection.row_count == EXPECTED_SOURCE_ROWS
pd.Series(
    {
        'path': str(AWARDS_CSV),
        'sha256': observed_sha256,
        'bytes': AWARDS_CSV.stat().st_size,
        'source_rows': inspection.row_count,
        'columns': inspection.column_count,
        'schema_sha256': inspection.ordered_schema_sha256,
    },
    name='pinned_input',
)

In [ ]:
SOURCE_FIELDS = [
    'Company',
    'Agency',
    'Phase',
    'Program',
    'Proposal Award Date',
    'Proposal Receipt Date',
    'Date of Notification',
    'Topic Code',
    'Award Year',
    'UEI',
    'Duns',
    'State',
    'PI Email',
    'RI Name',
]

canonical = load_sbir_awards_csv(AWARDS_CSV)
assert len(canonical) == EXPECTED_CANONICAL_AWARDS
source = pd.read_csv(
    AWARDS_CSV,
    usecols=SOURCE_FIELDS,
    dtype=str,
    keep_default_na=False,
    low_memory=False,
)
source['source_row'] = np.arange(2, len(source) + 2)
awards = canonical.merge(source, on='source_row', how='left', validate='one_to_one')
assert len(awards) == EXPECTED_CANONICAL_AWARDS

program = awards['program'].astype('string').str.strip().str.upper()
phase = awards['phase'].astype('string').str.strip().str.upper()
sttr = awards.loc[program.eq('STTR')].copy()
sttr_p1 = awards.loc[program.eq('STTR') & phase.eq('PHASE I')].copy()
pd.Series(
    {
        'canonical_awards': len(awards),
        'all_sttr_awards': len(sttr),
        'sttr_phase_i_awards': len(sttr_p1),
    }
)

## (a) RI-name field coverage

Count canonical awards, but recover `RI Name` from the source edition selected by the canonical collapse. Year spans mean at least one populated RI occurs in the year, not that every row in the year is populated.

In [ ]:
def nonblank(series: pd.Series) -> pd.Series:
    return series.fillna('').astype(str).str.strip().ne('')


def compact_years(values: pd.Series) -> str:
    years = sorted(pd.to_numeric(values, errors='coerce').dropna().astype(int).unique())
    if not years:
        return 'not present'
    runs: list[tuple[int, int]] = []
    start = prior = years[0]
    for year in years[1:]:
        if year == prior + 1:
            prior = year
            continue
        runs.append((start, prior))
        start = prior = year
    runs.append((start, prior))
    return '; '.join(str(start) if start == end else f'{start}–{end}' for start, end in runs)


coverage_rows = []
for agency, group in sttr.groupby('agency', dropna=False):
    populated = nonblank(group['RI Name'])
    coverage_rows.append(
        {
            'agency': agency,
            'sttr_awards': len(group),
            'ri_populated': int(populated.sum()),
            'share': float(populated.mean()),
            'populated_award_years': compact_years(group.loc[populated, 'Award Year']),
        }
    )
coverage = pd.DataFrame(coverage_rows).sort_values('sttr_awards', ascending=False)
coverage

## (d) Phase I counts by normalized RI string

Reuse the existing frozen STTR kernel's organization resolution behavior. Do not add aliases or inspect outcomes. A representative raw name is the most frequent spelling within a normalized key, with lexical tie-breaking.

In [ ]:
resolved_ri = sttr_p1['RI Name'].map(
    lambda value: resolve_identity(value, kind=IdentityKind.ORGANIZATION)
)
sttr_p1['_ri_key'] = resolved_ri.map(
    lambda identity: identity.normalized
    if identity is not None and identity.guard_passed
    else ''
)
guard_failures = int(
    resolved_ri.map(lambda identity: identity is not None and not identity.guard_passed).sum()
)
valid_ri = sttr_p1.loc[sttr_p1['_ri_key'].ne('')].copy()
ri_counts = valid_ri.groupby('_ri_key').size().rename('phase_i_awards')
thresholds = pd.Series(
    {
        'phase_i_awards': len(sttr_p1),
        'ri_populated': len(valid_ri),
        'blank_ri': int(sttr_p1['_ri_key'].eq('').sum()) - guard_failures,
        'guard_failures': guard_failures,
        'distinct_normalized_ri_strings': len(ri_counts),
        'ri_strings_ge_10': int(ri_counts.ge(10).sum()),
        'ri_strings_ge_30': int(ri_counts.ge(30).sum()),
        'ri_strings_ge_50': int(ri_counts.ge(50).sum()),
    }
)
thresholds

In [ ]:
label_counts = (
    valid_ri.groupby(['_ri_key', 'RI Name'])
    .size()
    .rename('label_count')
    .reset_index()
    .sort_values(['_ri_key', 'label_count', 'RI Name'], ascending=[True, False, True])
)
representatives = label_counts.drop_duplicates('_ri_key').set_index('_ri_key')['RI Name']
top_25 = (
    ri_counts.to_frame()
    .assign(representative_name=representatives)
    .sort_values(['phase_i_awards', 'representative_name'], ascending=[False, True])
    .head(25)
    .reset_index(drop=True)
)
top_25.index = np.arange(1, len(top_25) + 1)
top_25.index.name = 'rank'
top_25

In [ ]:
agency_ri_counts = valid_ri.groupby(['agency', '_ri_key']).size().rename('awards')
agency_ge_10 = (
    agency_ri_counts.loc[agency_ri_counts.ge(10)]
    .groupby(level='agency')
    .agg(ri_strings='size', covered_awards='sum')
    .sort_values('ri_strings', ascending=False)
)
cross_agency_ge_10 = int(
    agency_ri_counts.loc[agency_ri_counts.ge(10)]
    .groupby(level='_ri_key')
    .size()
    .ge(2)
    .sum()
)
display(agency_ge_10)
print('RI strings with >=10 Phase I awards in >=2 agencies:', cross_agency_ge_10)

## (g) PI email and (h) agreement-timing fields

The source has no PI employer/affiliation field and no allocation-of-rights agreement date. Count the nearby fields without treating them as the missing concepts.

In [ ]:
EMAIL_RE = re.compile(r'[A-Z0-9._%+\-]+@([A-Z0-9.\-]+)', flags=re.IGNORECASE)


def has_edu_domain(value: object) -> bool:
    domains = EMAIL_RE.findall(str(value or ''))
    return any(domain.lower().rstrip('.').endswith('.edu') for domain in domains)


pi_email_present = nonblank(sttr_p1['PI Email'])
edu_email = sttr_p1['PI Email'].map(has_edu_domain)
field_counts = pd.DataFrame(
    [
        {'field_or_proxy': 'PI Email populated', 'count': int(pi_email_present.sum())},
        {'field_or_proxy': 'any parsed .edu email domain', 'count': int(edu_email.sum())},
        {
            'field_or_proxy': 'Proposal Receipt Date populated',
            'count': int(nonblank(sttr_p1['Proposal Receipt Date']).sum()),
        },
        {
            'field_or_proxy': 'Date of Notification populated',
            'count': int(nonblank(sttr_p1['Date of Notification']).sum()),
        },
        {
            'field_or_proxy': 'Proposal Award Date populated',
            'count': int(nonblank(sttr_p1['Proposal Award Date']).sum()),
        },
    ]
).assign(share=lambda frame: frame['count'] / len(sttr_p1))
field_counts

## (i) Firms holding both SBIR and STTR

Reuse the repository's conservative firm key, exact topic-code normalizer, and government-FY helper. The topic calculation is explicitly a provisional exact-code proxy, not a topic-family grouping.

In [ ]:
firm_frame = awards.copy()
firm_frame['_source_record_number'] = firm_frame['source_row'].astype(int)
firm_frame = assign_firm_keys(firm_frame)
firm_frame['_program_key'] = firm_frame['Program'].fillna('').str.strip().str.upper()
firm_frame['_agency_key'] = firm_frame['Agency'].fillna('').str.strip()
firm_frame['_topic_key'] = firm_frame['Topic Code'].map(normalize_topic_code)
firm_frame['_source_award_year'] = pd.to_numeric(
    firm_frame['Award Year'], errors='coerce'
).astype('Int64')
fy_helper = FiscalShockAggregator(config=object())  # method needs no project config
# Call the helper on its canonical lower-case contract. The raw blank-string
# `Proposal Award Date` is retained only for the field-coverage audit above.
firm_frame['_fiscal_year'] = firm_frame[['award_date', 'award_year']].apply(
    fy_helper.extract_fiscal_year, axis=1
)


def mixed_program_count(frame: pd.DataFrame, dimensions: list[str]) -> dict[str, int]:
    eligible = frame.loc[frame['_program_key'].isin({'SBIR', 'STTR'})].copy()
    for dimension in dimensions:
        eligible = eligible.loc[eligible[dimension].notna()]
        if eligible[dimension].dtype == object or pd.api.types.is_string_dtype(
            eligible[dimension]
        ):
            eligible = eligible.loc[eligible[dimension].astype(str).str.strip().ne('')]
    grouped = eligible.groupby(['_firm_key', *dimensions])['_program_key'].agg(set)
    mixed = grouped.map(lambda values: {'SBIR', 'STTR'}.issubset(values))
    cells = grouped.loc[mixed].index
    return {
        'firms': int(cells.get_level_values('_firm_key').nunique()),
        'cells': len(cells),
    }


phase_i_frame = firm_frame.loc[
    firm_frame['Phase'].fillna('').str.strip().str.upper().eq('PHASE I')
]
mixed_program_summary = pd.DataFrame(
    {
        'all_phases_same_agency': mixed_program_count(firm_frame, ['_agency_key']),
        'all_phases_fy_exact_topic': mixed_program_count(
            firm_frame, ['_agency_key', '_fiscal_year', '_topic_key']
        ),
        'all_phases_source_year_exact_topic': mixed_program_count(
            firm_frame, ['_agency_key', '_source_award_year', '_topic_key']
        ),
        'phase_i_same_agency': mixed_program_count(phase_i_frame, ['_agency_key']),
        'phase_i_fy_exact_topic': mixed_program_count(
            phase_i_frame, ['_agency_key', '_fiscal_year', '_topic_key']
        ),
    }
).T
display(mixed_program_summary)
print(
    'fiscal-year coverage:',
    int(firm_frame['_fiscal_year'].notna().sum()),
    'of',
    len(firm_frame),
)

## (b), (c), and (f) named artifact availability

These existence checks do not turn a legacy or gated file into an admissible outcome. Interpretation belongs in the Phase 0 readout.

In [ ]:
named_artifacts = {
    'existing_sttr_spec': REPO_ROOT / 'specs/sttr-spinout-linkage/design.md',
    'ipeds_with_cbsa_source': REPO_ROOT
    / 'data/reference/sttr_partner_type_seed_lists/ipeds_institutions/HD2024.csv',
    'form_d_company_records': REPO_ROOT / 'data/form_d_details.jsonl',
    'efts_scan': REPO_ROOT / 'data/sec_edgar_scan.jsonl',
    'current_ma_direction_refinement': REPO_ROOT / 'data/sbir_ma_direction_refined.jsonl',
    'coded_phase_iii_survival': REPO_ROOT
    / 'data/processed/phase_transition_survival.parquet',
    'coded_phase_iii_pairs': REPO_ROOT / 'data/processed/phase_ii_iii_pairs.parquet',
}
pd.DataFrame(
    [
        {'artifact': name, 'path': str(path), 'exists_on_host': path.exists()}
        for name, path in named_artifacts.items()
    ]
)

## Findings and stop condition

| Feasibility statement | Evidence | Limitation |
|---|---|---|
| Repeated normalized RI strings exist at useful overall counts | `thresholds`, `top_25` | Strings are not resolved institutions |
| Sparse-market counts cannot be produced | repository inventory in the linked readout | No canonical RI→CBSA or Form D density primitive |
| A `.edu` PI-email sensitivity is possible | `field_counts` | It is not employer/affiliation |
| Same-agency mixed-program firms exist | `mixed_program_summary` | Exact topic code is not a topic family |

STOP after Phase 0. Do not estimate RI variance, form density tiers, adjust name matching after looking at outcomes, or run any extension until a reviewed Phase 1 spec is frozen. This notebook remains exploratory and non-citable.